# Moduł 16 — Bielik-11B-v3.0-Instruct: Praktyczny przewodnik

Na **2. etapie III Olimpiady AI** (13–15 marca 2026) dostępny będzie model **Bielik-11B-v3.0-Instruct**. 
Ten moduł uczy Cię jak go efektywnie używać w kontekście zawodów.

## Co to jest Bielik?
| Cecha | Wartość |
|---|---|
| Pełna nazwa | `speakleash/Bielik-11B-v3.0-Instruct` |
| Parametry | 11 miliardów |
| Format czatu | ChatML (`<\|im_start\|>`, `<\|im_end\|>`) |
| Licencja | Apache 2.0 |
| VRAM (bfloat16) | ~22 GB |
| Twórcy | SpeakLeash & ACK Cyfronet AGH |
| Język | Polski i angielski (dominacja PL) |

## Kiedy używać Bielika na olimpiadzie?

 **Warto użyć gdy:**
- Zadanie wymaga analizy/generacji **tekstu po polsku**
- Potrzebujesz **klasyfikacji tekstu** (sentiment, temat, kategoria)
- Zadanie wymaga **ekstrakcji informacji** z tekstu
- Chcesz wygenerować **wyjaśnienia** lub **opisy**
- Potrzebujesz **podsumowania** dokumentów
- Zadanie z **tłumaczeniem** PL↔EN

 **Nie używaj gdy:**
- Zadanie jest czysto **wizyjne** (CNN/segmentacja) — nie trać czasu GPU
- Wystarczy klasyczny ML (XGBoost, RandomForest) — szybsze i pewniejsze
- Zadanie wymaga tylko obliczeń numerycznych (numpy/torch wystarczy)
- Masz bardzo mało czasu, a zadanie nie wymaga NLP

 **Pamiętaj:** Bielik zużywa dużo VRAM. Jeśli zadanie nie wymaga LLM — nie ładuj go.

## 1. Ładowanie modelu

Na olimpiadzie model będzie dostępny lokalnie (bez internetu). 
Poniżej standardowy sposób ładowania z `transformers`.

In [ ]:
# UWAGA: Ten kod wymaga GPU z ~22 GB VRAM (np. A100)
# Na zwykłym laptopie NIE uruchamiaj — przeczytaj i zrozum wzorzec

# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch

# MODEL_NAME = "speakleash/Bielik-11B-v3.0-Instruct"

# # Ładowanie tokenizera
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# # Ładowanie modelu w bfloat16 (połowa pamięci vs float32)
# model = AutoModelForCausalLM.from_pretrained(
# MODEL_NAME,
# torch_dtype=torch.bfloat16,
# device_map="auto" # Automatycznie rozłoży na dostępne GPU
# )

# print(f"Model załadowany! Parametry: {sum(p.numel() for p in model.parameters()):,}")
# print(f"VRAM: ~{torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 2. Format ChatML — klucz do komunikacji z Bielikiem

Bielik używa formatu **ChatML** (jak OpenAI). Każda wiadomość:

```
<|im_start|> rola
treść wiadomości<|im_end|>
```

Role: `system`, `user`, `assistant`.

In [ ]:
# Format ChatML — ćwiczenie offline (bez GPU)

def format_chatml(messages):
 """
 Formatuj listę wiadomości w ChatML.
 Przydatne gdybyś musiał ręcznie budować prompt.
 """
 prompt = "<s>"
 for msg in messages:
 prompt += f"<|im_start|> {msg['role']}\n{msg['content']}<|im_end|> \n"
 prompt += "<|im_start|> assistant\n"
 return prompt

# Przykład 1: Proste pytanie
messages_simple = [
 {"role": "system", "content": "Odpowiadaj krótko po polsku."},
 {"role": "user", "content": "Co to jest sieć neuronowa?"}
]

print("=== Prosty prompt ===")
print(format_chatml(messages_simple))
print()

# Przykład 2: Klasyfikacja tekstu (częsty wzorzec olimpiadowy)
messages_classify = [
 {"role": "system", "content": "Jesteś klasyfikatorem sentymentu. Odpowiadaj TYLKO jednym słowem: POZYTYWNY, NEGATYWNY lub NEUTRALNY."},
 {"role": "user", "content": "Sklasyfikuj sentyment: 'Ten produkt jest absolutnie fantastyczny!'"}
]

print("=== Klasyfikacja sentymentu ===")
print(format_chatml(messages_classify))
print()

# Przykład 3: Ekstrakcja informacji (wzorzec z II OAI)
messages_extract = [
 {"role": "system", "content": "Wyodrębnij z tekstu: imię, zawód, miasto. Odpowiedz w formacie JSON."},
 {"role": "user", "content": "Tekst: 'Jan Kowalski jest programistą mieszkającym w Krakowie.'"}
]

print("=== Ekstrakcja informacji ===")
print(format_chatml(messages_extract))

## 3. Generowanie odpowiedzi — wzorzec olimpiadowy

Na olimpiadzie najpewniej użyjesz `tokenizer.apply_chat_template()` i `model.generate()`. 
Poniżej pełny wzorzec do zapamiętania.

In [ ]:
# WZORZEC GENEROWANIA — zapamiętaj ten schemat!
# (zakomentowany — wymaga GPU)

def generate_bielik_response(messages, model, tokenizer, max_new_tokens=200, temperature=0.1):
 """
 Generuj odpowiedź z Bielika.

 Args:
 messages: Lista słowników [{"role": ..., "content": ...}]
 model: Załadowany model
 tokenizer: Załadowany tokenizer
 max_new_tokens: Maks. długość odpowiedzi
 temperature: 0.0=deterministyczny, 1.0=kreatywny
 """
 # 1. Tokenizacja z ChatML template
 input_ids = tokenizer.apply_chat_template(
 messages,
 return_tensors="pt",
 add_generation_prompt=True # Dodaje <|im_start|> assistant
 ).to(model.device)

 # 2. Generowanie
 with torch.no_grad():
 output_ids = model.generate(
 input_ids,
 max_new_tokens=max_new_tokens,
 temperature=temperature,
 do_sample=temperature > 0,
 top_p=0.95 if temperature > 0 else 1.0,
 repetition_penalty=1.1,
 eos_token_id=tokenizer.eos_token_id,
 pad_token_id=tokenizer.eos_token_id,
 )

 # 3. Dekodowanie TYLKO nowych tokenów
 new_tokens = output_ids[0][input_ids.shape[1]:]
 response = tokenizer.decode(new_tokens, skip_special_tokens=True)
 return response.strip()

# Przykład użycia (zakomentowany — wymaga GPU):
# messages = [
# {"role": "system", "content": "Odpowiadaj krótko."},
# {"role": "user", "content": "Wymień 3 zastosowania sieci neuronowych."}
# ]
# response = generate_bielik_response(messages, model, tokenizer)
# print(response)

print(" Funkcja generate_bielik_response() gotowa — zapamiętaj wzorzec!")
print("\nKluczowe parametry:")
print(" temperature=0.1 → prawie deterministyczny (klasyfikacja)")
print(" temperature=0.7 → kreatywny (generacja tekstu)")
print(" max_new_tokens=50 → krótka odpowiedź (klasa, etykieta)")
print(" max_new_tokens=500 → dłuższa odpowiedź (opis, streszczenie)")

## 4. Zastosowania olimpiadowe — gotowe wzorce

Poniżej 5 najważniejszych scenariuszy z konkretnymi promptami.

In [ ]:
# === WZORZEC 1: Klasyfikacja tekstu ===
# Zadanie: Przyporządkuj tekst do jednej z kategorii
# Wzorzec z: II OAI (Halucynuj), III OAI (Wieloetykietowa klasyfikacja)

def make_classification_prompt(text, categories, allow_multiple=False):
 """Generuj prompt do klasyfikacji tekstu przez Bielika."""
 cat_str = ', '.join(categories)
 if allow_multiple:
 instruction = f"""Sklasyfikuj poniższy tekst. Wybierz WSZYSTKIE pasujące kategorie z listy: [{cat_str}].
Odpowiedz TYLKO listą kategorii oddzielonych przecinkami, bez dodatkowych komentarzy."""
 else:
 instruction = f"""Sklasyfikuj poniższy tekst do DOKŁADNIE JEDNEJ kategorii z listy: [{cat_str}].
Odpowiedz TYLKO nazwą kategorii, bez dodatkowych komentarzy."""

 return [
 {"role": "system", "content": instruction},
 {"role": "user", "content": f"Tekst: '{text}'"}
 ]

# Przykład
categories = ["sport", "polityka", "technologia", "nauka", "kultura"]
text = "Nowy procesor Intela osiąga rekordowe wyniki w benchmarkach AI"
prompt = make_classification_prompt(text, categories)
print("Prompt klasyfikacji:")
for msg in prompt:
 print(f" [{msg['role']}]: {msg['content'][:80]}...")

print("\n" + "="*60)

# === WZORZEC 2: Ekstrakcja informacji (NER) ===
# Wzorzec z: II OAI (Ekstrakcja źródeł)

def make_extraction_prompt(text, fields):
 """Prompt do ekstrakcji ustrukturyzowanych danych."""
 fields_str = ', '.join(fields)
 return [
 {"role": "system", "content": f"""Wyodrębnij z tekstu następujące pola: {fields_str}.
Odpowiedz w formacie JSON. Jeśli pole nie występuje w tekście, ustaw wartość null."""},
 {"role": "user", "content": text}
 ]

text_ner = "Dr Anna Nowak z Politechniki Wrocławskiej opublikowała artykuł o kwantyzacji modeli w 2025 roku."
fields = ["osoba", "instytucja", "temat", "rok"]
prompt_ner = make_extraction_prompt(text_ner, fields)
print("\nPrompt ekstrakcji:")
for msg in prompt_ner:
 print(f" [{msg['role']}]: {msg['content'][:80]}...")

print("\n" + "="*60)

# === WZORZEC 3: Podsumowanie ===
# Przydatne do streszczania dłuższych tekstów

def make_summary_prompt(text, max_sentences=3):
 """Prompt do podsumowania tekstu."""
 return [
 {"role": "system", "content": f"Podsumuj tekst w maksymalnie {max_sentences} zdaniach. Bądź zwięzły i precyzyjny."},
 {"role": "user", "content": text}
 ]

print("\nWzorzec podsumowania — gotowy do użycia z generate_bielik_response()")

print("\n" + "="*60)

# === WZORZEC 4: Tłumaczenie ===
# Wzorzec z: I OAI finał (Tłumaczenie maszynowe), II OAI finał (Stylizacja tłumaczenia)

def make_translation_prompt(text, source_lang="polski", target_lang="angielski", style=None):
 """Prompt do tłumaczenia z opcjonalnym stylem."""
 if style:
 instruction = f"Przetłumacz tekst z {source_lang} na {target_lang} w stylu: {style}. Podaj TYLKO tłumaczenie."
 else:
 instruction = f"Przetłumacz tekst z {source_lang} na {target_lang}. Podaj TYLKO tłumaczenie."
 return [
 {"role": "system", "content": instruction},
 {"role": "user", "content": text}
 ]

prompt_translate = make_translation_prompt("Sieci neuronowe są fascynujące.", style="formalny akademicki")
print("\nPrompt tłumaczenia ze stylizacją (II OAI finał):")
for msg in prompt_translate:
 print(f" [{msg['role']}]: {msg['content']}")

print("\n" + "="*60)

# === WZORZEC 5: Parsowanie i analiza danych ===
# Przydatne gdy Bielik dostaje dane i musi je przetworzyć

def make_analysis_prompt(data_description, data, question):
 """Prompt do analizy danych tekstowych."""
 return [
 {"role": "system", "content": f"Jesteś analitykiem danych. Analizujesz: {data_description}. Odpowiadaj precyzyjnie."},
 {"role": "user", "content": f"Dane:\n{data}\n\nPytanie: {question}"}
 ]

print("\n Wszystkie 5 wzorców gotowe do użycia na olimpiadzie!")

## 5. Zarządzanie pamięcią GPU

Na olimpiadzie będziesz miał GPU (T4 lub A100). Bielik w bfloat16 zajmuje ~22 GB VRAM. 
Musisz umieć **zwalniać pamięć** gdy skończysz z modelem.

In [ ]:
import torch
import gc

# Sprawdzanie dostępnej pamięci GPU
if torch.cuda.is_available():
 print(f"GPU: {torch.cuda.get_device_name(0)}")
 print(f"VRAM total: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
 print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
 print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated()) / 1e9:.1f} GB")
else:
 print("Brak GPU — Bielik wymaga CUDA GPU z min. 22 GB VRAM")

def free_gpu_memory(model=None, tokenizer=None):
 """
 Zwolnij pamięć GPU po zakończeniu pracy z Bielikiem.
 WAŻNE: Wywołaj to zanim załadujesz inny model (np. CNN)!
 """
 if model is not None:
 del model
 if tokenizer is not None:
 del tokenizer
 gc.collect()
 if torch.cuda.is_available():
 torch.cuda.empty_cache()
 print(f"Pamięć po zwolnieniu: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

print("\n Strategia GPU na olimpiadzie:")
print("1. Jeśli zadanie wymaga Bielika → załaduj go PIERWSZE")
print("2. Użyj Bielika do wszystkich zdań NLP naraz (batch)")
print("3. Zapisz wyniki do zmiennej/pliku")
print("4. Zwolnij pamięć: free_gpu_memory(model, tokenizer)")
print("5. Teraz możesz załadować CNN/inny model")

## 6. Bielik jako narzędzie wspomagające — nie rozwiązanie samo w sobie

Kluczowy insight: Bielik to **narzędzie** w twoim arsenale, nie magiczne rozwiązanie.

### Scenariusze hybrydowe (najczęstsze na olimpiadzie)

| Scenariusz | Bielik | Klasyczny ML/DL |
|---|---|---|
| Klasyfikacja tekstu | Generuje cechy / etykiety | Ostateczny klasyfikator (XGBoost) |
| Analiza sentymentu | Bezpośrednia klasyfikacja | Walidacja wyników |
| Ekstrakcja cech z tekstu | Embedddingi z ukrytych warstw | Clustering (KMeans) |
| Augmentacja danych | Parafrazowanie zdań | Trening na powiększonym zbiorze |
| Tłumaczenie + styl | Tłumaczenie z formatowaniem | Metryka BLEU do oceny |

In [ ]:
# === SCENARIUSZ HYBRYDOWY: Bielik + Klasyczny ML ===
# Wzorzec: Użyj Bielika do wygenerowania cech, potem XGBoost do klasyfikacji

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Symulacja: Bielik wygenerował embeddingi (w praktyce z ukrytych warstw)
np.random.seed(42)
n_samples = 200
n_features = 64 # Zredukowane embeddingi

# Klasa 0: pozytywne recenzje, Klasa 1: negatywne
X_pos = np.random.randn(n_samples // 2, n_features) + 0.5
X_neg = np.random.randn(n_samples // 2, n_features) - 0.5
X = np.vstack([X_pos, X_neg])
y = np.array([0] * (n_samples // 2) + [1] * (n_samples // 2))

# Shuffle
idx = np.random.permutation(n_samples)
X, y = X[idx], y[idx]

# Split
split = int(0.8 * n_samples)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Klasyczny ML na embeddingach z Bielika
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("=== Hybrydowy pipeline: Bielik (embeddingi) → RandomForest ===")
print(classification_report(y_test, y_pred, target_names=["pozytywne", "negatywne"]))

print("\n Ten wzorzec łączy siłę LLM (rozumienie języka) z niezawodnością ML (klasyfikacja)")

## 7. Przetwarzanie batch — oszczędność czasu

Na olimpiadzie czas jest ograniczony. Zamiast generować odpowiedź dla każdego tekstu osobno, 
przetwarzaj dane **partiami** (batch).

In [ ]:
import time

# === Batch processing pattern ===

def batch_classify_with_bielik(texts, categories, model=None, tokenizer=None):
 """
 Klasyfikuj wiele tekstów na raz.
 W prawdziwym użyciu: model i tokenizer to załadowany Bielik.
 Tu: symulacja z losowym przypisaniem.
 """
 results = []

 for i, text in enumerate(texts):
 messages = [
 {"role": "system", "content": f"Sklasyfikuj tekst do jednej kategorii: {', '.join(categories)}. Odpowiedz TYLKO nazwą kategorii."},
 {"role": "user", "content": text}
 ]

 # W prawdziwym użyciu:
 # response = generate_bielik_response(messages, model, tokenizer, max_new_tokens=20)
 # results.append(response.strip())

 # Symulacja:
 import random
 results.append(random.choice(categories))

 return results

# Symulacja batch processing
texts = [
 "Polska wygrała mecz z Niemcami 3:1",
 "Nowy model GPT-5 przewyższa poprzednika",
 "Sejm uchwalił nową ustawę",
 "Odkryto nową planetę w Układzie Słonecznym",
 "Film Nolana zdobył 7 Oscarów",
]

categories = ["sport", "technologia", "polityka", "nauka", "kultura"]

start = time.time()
predictions = batch_classify_with_bielik(texts, categories)
elapsed = time.time() - start

print("=== Batch Classification Results ===")
for text, pred in zip(texts, predictions):
 print(f" '{text[:40]}...' → {pred}")
print(f"\nCzas: {elapsed:.2f}s dla {len(texts)} tekstów")
print("\n Na GPU: ~2-5s per tekst. Planuj czas i nie ładuj modelu wielokrotnie!")

## 8. Prompt Engineering — mikro-optymalizacje

Jakość promptu ma **ogromny** wpływ na wynik. Kilka zasad:

In [ ]:
# === Zasady Prompt Engineering dla Bielika ===

print("=" * 60)
print("ZASADY PROMPT ENGINEERING DLA BIELIKA NA OLIMPIADZIE")
print("=" * 60)

rules = {
 "1. Bądź precyzyjny w system prompt": {
 " Źle": "Odpowiedz na pytanie.",
 " Dobrze": "Odpowiedz JEDNYM SŁOWEM: POZYTYWNY lub NEGATYWNY."
 },
 "2. Ogranicz format odpowiedzi": {
 " Źle": "Podaj wynik.",
 " Dobrze": "Odpowiedz TYLKO liczbą zmiennoprzecinkową, np. 0.85"
 },
 "3. Podawaj przykłady (few-shot)": {
 " Źle": "Sklasyfikuj tekst.",
 " Dobrze": "Przykłady:\n'Super film' → POZYTYWNY\n'Okropny' → NEGATYWNY\nTeraz sklasyfikuj: ..."
 },
 "4. Mów po polsku do Bielika": {
 " Źle": "Classify the following text",
 " Dobrze": "Sklasyfikuj poniższy tekst"
 },
 "5. Ustaw niską temperaturę dla zadań deterministycznych": {
 "Klasyfikacja": "temperature=0.1",
 "Ekstrakcja": "temperature=0.1",
 "Generacja kreatywna": "temperature=0.7",
 "Tłumaczenie": "temperature=0.3"
 }
}

for rule, examples in rules.items():
 print(f"\n{rule}:")
 for key, value in examples.items():
 print(f" {key}: {value}")

# === Few-shot prompt template ===
print("\n" + "=" * 60)
print("\n WZORZEC FEW-SHOT (5 przykładów):")

def make_few_shot_prompt(examples, test_input, task_description):
 """
 Generuj few-shot prompt.
 examples: lista (input, output) par
 """
 examples_str = "\n".join([f"Wejście: {inp} → Wynik: {out}" for inp, out in examples])
 return [
 {"role": "system", "content": f"{task_description}\n\nPrzykłady:\n{examples_str}\n\nOdpowiedz TYLKO wynikiem."},
 {"role": "user", "content": f"Wejście: {test_input}"}
 ]

examples = [
 ("Świetny produkt!", "POZYTYWNY"),
 ("Nie polecam.", "NEGATYWNY"),
 ("W porządku.", "NEUTRALNY"),
]
prompt_fs = make_few_shot_prompt(examples, "Absolutnie genialny!", "Klasyfikacja sentymentu")
print(format_chatml(prompt_fs))

## 9. Podsumowanie — checklista na olimpiadę

### Przed olimpiadą zapamiętaj:

1. **Ładowanie**: `AutoModelForCausalLM.from_pretrained(..., torch_dtype=torch.bfloat16, device_map="auto")`
2. **Tokenizacja**: `tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)`
3. **Generowanie**: `model.generate(input_ids, max_new_tokens=200, temperature=0.1)`
4. **Dekodowanie**: `tokenizer.decode(output[input_len:], skip_special_tokens=True)`
5. **Zwalnianie**: `del model; gc.collect(); torch.cuda.empty_cache()`

### Wzorzec ChatML:
```
<|im_start|> system
Instrukcja...<|im_end|>
<|im_start|> user
Pytanie...<|im_end|>
<|im_start|> assistant
```

### Kluczowe parametry:
- `temperature=0.1` — klasyfikacja, ekstrakcja
- `temperature=0.7` — generacja kreatywna
- `max_new_tokens` — kontroluj długość (50 dla etykiet, 500 dla opisów)
- `repetition_penalty=1.1` — zapobiega powtórzeniom

### Kiedy NIE ładować Bielika:
- Zadanie czysto wizyjne (CNN)
- Wystarczy scikit-learn/XGBoost
- Zostało < 30 min i nie potrzebujesz NLP